In [2]:
def sq(x): return x**2

In [4]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import numpy as np

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("first-test")
    .master("local[*]")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)

sc = spark.sparkContext

In [ ]:
data = np.random.rand(100).astype('float32')
rdd = sc.parallelize(data)

avg_accumulator = (0.0, 0.0)

def mergeValue(acc, value): return (acc[0] + value, acc[1] + 1)
# merge by summing partial_sums and partial_counts
def mergeAccum(acc1, acc2): return (acc1[0] + acc2[0], acc1[1] + acc2[1])
# spark aggregate
sum_and_count = rdd.aggregate(avg_accumulator, mergeValue, mergeAccum)

# mean = rdd.mean()
# var = rdd.foreach(lambda x: )
avg = sum_and_count[0] / sum_and_count[1]

shared_avg = sc.broadcast(avg)

def mergeValueVar(acc, value): return ((value - shared_avg.value)**2 + acc)

def mergeAccumVar(acc1, acc2): return (acc1 + acc2)

variance = rdd.aggregate(0.0, mergeValueVar, mergeAccumVar)
print(rdd.variance())
print ("Average is", avg)
print ("Variance is", variance / sum_and_count[1])

0.091579564
Average is 0.4940017
Variance is 0.09157955


In [6]:
poems = sc.textFile("poems.txt")
print(poems.take(10))


['', '', '', '\tSONNETS', '', '', '', 'TO THE ONLY BEGETTER OF', 'THESE INSUING SONNETS', 'MR. W. H. ALL HAPPINESS']


In [ ]:
words = ( # parentheses are used just for indentation
    # get words from lines
    poems.flatMap( lambda line: line.split() )
    # crate (word,1) pairs
    .map( lambda word: (word,1) )
    # sum up occurrences
    .reduceByKey( lambda count1,count2: count1+count2 )
    # remove infrequent
    .filter( lambda wc_pair: wc_pair[1]>=600 )
    # reverse pairs
    .map( lambda wc_pair: (wc_pair[1], wc_pair[0]) )
)
words.top(10)

['', '', '', '\tSONNETS', '', '', '', 'TO THE ONLY BEGETTER OF', 'THESE INSUING SONNETS', 'MR. W. H. ALL HAPPINESS']


[(1246, 'the'), (920, 'to'), (826, 'and'), (812, 'of'), (742, 'in')]

In [ ]:
accumulator = (0, 0)

average_length = ( # parentheses are used just for indentation
    poems.flatMap( lambda line: line.split() )
    .map( lambda word: (len(word), 1) )
    .aggregate(
        zeroValue=accumulator,
        seqOp=lambda acc, word: (acc[0] + word[0], acc[1] + word[1]),
        combOp=lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
    )
)
print("Average word length:", average_length[0] / average_length[1])

Average word length: 4.475226567477035


In [8]:
from operator import add
# word lenght distribution
length_distribution = (
    poems.flatMap( lambda line: line.split() )
    .map( lambda word: (len(word), 1) )
    .reduceByKey(add)
    .map( lambda wc_pair: (wc_pair[1], wc_pair[0]))
    .sortByKey(ascending=False)
)
print(length_distribution.take(10))

[(10287, 4), (9499, 3), (7597, 5), (7175, 2), (5087, 6), (3326, 7), (2097, 8), (1229, 9), (1184, 1), (596, 10)]


In [ ]:
alphabet_distribution = (
    poems.flatMap( lambda line: line.split() )
    .map( lambda word: (word[0].lower(), 1) )
    .reduceByKey( lambda count1,count2: count1+count2)
    .map( lambda wc_pair: (wc_pair[1], wc_pair[0]))
    .sortByKey(ascending=False)
)

print(alphabet_distribution.take(10))

[(7746, 't'), (4379, 's'), (4103, 'a'), (3465, 'w'), (3324, 'h'), (2842, 'b'), (2676, 'i'), (2610, 'm'), (2243, 'f'), (2120, 'o')]


In [ ]:
# coocciring

def get_pairs(words):
    pairs = []
    for i in range(len(words) - 1):
        pairs.append((words[i], words[i + 1]))
    return pairs

cooccurring = (
    poems.flatMap(lambda line: line.split("\t"))
    .map(lambda line: (line[0].lower().split(), line[1]))
    .map(lambda word_line: (get_pairs(word_line[0]), word_line[1]))
)
print(cooccurring.take(10))

[([], 0), ([], 1), ([], 2), ([], 3), ([], 4), ([], 5), ([], 6), ([('to', 'the'), ('the', 'only'), ('only', 'begetter'), ('begetter', 'of')], 7), ([('these', 'insuing'), ('insuing', 'sonnets')], 8), ([('mr.', 'w.'), ('w.', 'h.'), ('h.', 'all'), ('all', 'happiness')], 9)]


In [ ]:
from operator import add

def adjacent_pairs(words):
    return [((words[i], words[i + 1]), 1) for i in range(len(words) - 1)]

pair_counts = (
    poems
    .map(lambda line: line.lower().split())
    .filter(lambda words: len(words) > 1)
    .flatMap(adjacent_pairs)
    .reduceByKey(add)
    .sortBy(lambda x: x[1], ascending=False)
)

print(pair_counts.take(20))

[(('in', 'the'), 87), (('to', 'the'), 75), (('in', 'his'), 51), (('like', 'a'), 49), (('of', 'the'), 47), (('in', 'my'), 46), (('of', 'his'), 40), (('in', 'thy'), 40), (('thou', 'art'), 39), (('to', 'be'), 39), (('of', 'my'), 39), (('in', 'her'), 37), (('that', 'i'), 36), (('and', 'in'), 34), (('my', 'love'), 33), (('on', 'the'), 32), (('from', 'the'), 32), (('i', 'have'), 31), (('of', 'thy'), 29), (('that', 'which'), 29)]


In [ ]:
def adjacent_pairs(words):
    return [((words[i], words[i + 1]), 1) for i in range(len(words) - 1)]

pair_counts = (
    poems
    .map(lambda line: line.lower().split())
    .filter(lambda words: len(words) > 0)

    # .map(lambda word: (word, 1))
    # .reduceByKey(lambda word1, word2: (word1, word2))
    # .flatMap(lambda words: words)
    # .filter(lambda words: len(words) > 1)
    # .flatMap(adjacent_pairs)
    # .reduceByKey(add)
    # .sortBy(lambda x: x[1], ascending=False)
)

print(pair_counts.take(20))

[['sonnets'], ['to', 'the', 'only', 'begetter', 'of'], ['these', 'insuing', 'sonnets'], ['mr.', 'w.', 'h.', 'all', 'happiness'], ['and', 'that', 'eternity'], ['promised', 'by'], ['our', 'ever-living', 'poet', 'wisheth'], ['the', 'well-wishing'], ['adventurer', 'in'], ['setting', 'forth'], ['t.', 't.'], ['i.'], ['from', 'fairest', 'creatures', 'we', 'desire', 'increase,'], ['that', 'thereby', "beauty's", 'rose', 'might', 'never', 'die,'], ['but', 'as', 'the', 'riper', 'should', 'by', 'time', 'decease,'], ['his', 'tender', 'heir', 'might', 'bear', 'his', 'memory:'], ['but', 'thou,', 'contracted', 'to', 'thine', 'own', 'bright', 'eyes,'], ["feed'st", 'thy', "light'st", 'flame', 'with', 'self-substantial', 'fuel,'], ['making', 'a', 'famine', 'where', 'abundance', 'lies,'], ['thyself', 'thy', 'foe,', 'to', 'thy', 'sweet', 'self', 'too', 'cruel.']]


In [27]:
import re
from operator import add

def tokenize(line):
    return re.findall(r"\w+", line.lower())

def adjacent_pairs(words):
    return [((words[i], words[i+1]), 1) for i in range(len(words)-1)]

# (index, words) for non-empty lines
words_idx = poems.map(lambda line: tokenize(line)) \
                 .zipWithIndex() \
                 .map(lambda lw: (lw[1], lw[0]))  # (idx, words)

# within-line adjacent pairs
within = words_idx.map(lambda kv: kv[1]) \
                  .flatMap(lambda ws: adjacent_pairs(ws) if len(ws) > 1 else [])

# across-line pairs: join previous line with current line
prev = words_idx.map(lambda kv: (kv[0] + 1, kv[1]))   # key = next index
joined = prev.join(words_idx)  # (idx, (prev_words, curr_words))
across = joined.flatMap(lambda kv: [(((kv[1][0][-1], kv[1][1][0]), 1))] 
                        if (len(kv[1][0]) > 0 and len(kv[1][1]) > 0) else [])

# union and aggregate counts
pair_counts = within.union(across).reduceByKey(add).sortBy(lambda x: x[1], ascending=False)

print(pair_counts.take(20))

[(('in', 'the'), 87), (('to', 'the'), 75), (('my', 'love'), 58), (('in', 'his'), 52), (('thou', 'art'), 49), (('like', 'a'), 49), (('love', 's'), 48), (('d', 'with'), 48), (('of', 'the'), 47), (('in', 'my'), 46), (('to', 'be'), 45), (('of', 'his'), 41), (('the', 'world'), 40), (('in', 'thy'), 40), (('of', 'my'), 39), (('i', 'have'), 38), (('in', 'her'), 37), (('that', 'i'), 36), (('my', 'heart'), 36), (('d', 'in'), 35)]


In [35]:
def all_pairs(words):
    pairs = []
    for i in range(len(words)):
        for j in range(len(words)):
            if i != j:
                pairs.append(((words[i], words[j]), 1))
    return pairs

line_words = (poems
    .map(lambda line: line.lower().split())
    .filter(lambda words: len(words) > 1)
    .flatMap(all_pairs)
    .reduceByKey(add)
    .sortBy(lambda x: x[1], ascending=False)
)

print(line_words.take(25))

[(('the', 'of'), 384), (('of', 'the'), 384), (('the', 'the'), 304), (('to', 'the'), 287), (('the', 'to'), 287), (('and', 'the'), 283), (('the', 'and'), 283), (('and', 'to'), 235), (('to', 'and'), 235), (('that', 'the'), 178), (('the', 'that'), 178), (('in', 'the'), 178), (('the', 'in'), 178), (('and', 'of'), 176), (('of', 'and'), 176), (('and', 'in'), 176), (('in', 'and'), 176), (('to', 'to'), 144), (('his', 'and'), 136), (('and', 'his'), 136), (('and', 'with'), 133), (('with', 'and'), 133), (('my', 'my'), 128), (('my', 'and'), 128), (('and', 'my'), 128)]


In [40]:
accumulator = (0, 0)

avg = (
    poems.flatMap( lambda line: line.split() )
    .map( lambda word: (len(word), 1) )
    .aggregate(
        zeroValue=accumulator,
        seqOp=lambda acc, word: (acc[0] + word[0], acc[1] + word[1]),
        combOp=lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
    )
)

print("word count:", avg[1])

print("Average word length:", avg[0] / avg[1])
# print("actual avg", poems.stats())
avg_ = avg[0] / avg[1]

global_avg = sc.broadcast(avg_)

accumulator = (0, 0)

std_dev = (
    poems.flatMap( lambda line: line.split() )
    .map( lambda word: (len(word), 1) )
    .aggregate(
        zeroValue=accumulator,
        seqOp=lambda acc, word: (acc[0] + (word[0] - global_avg.value)**2, acc[1] + 1),
        combOp=lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
    )
)

print("Standard deviation of word length:", (std_dev[0] / std_dev[1])**0.5)



word count: 48661
Average word length: 4.475226567477035
Standard deviation of word length: 2.1256698717130798


In [ ]:
spark.stop()